# Notebook 03 — Mechanism ablations and protocol freeze

**Purpose.** Byte- and exposure-matched ablations A1 (foveation), A2 (learned gate), P (disagreement auxiliary) and the exploratory P+MSF; comparator selection on tune; development paired-loss SD and MDE scenarios; leakage audit rerun; protocol lock. **Partitions:** train, tune. **GPU:** ablation pool. This notebook may reject the proposed method; rejection is a valid result.

In [1]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


repo: $CAPE_ROOT/cape-eeg
workspace root: $CAPE_ROOT
data root: $CAPE_ROOT
private: $CAPE_ROOT/private


## Ablation fits (seed 101, identical schedule and augmentation)

In [2]:
for cfg in ['A1', 'A2', 'P', 'P_MSF']:
    run(['train.py', '--config', cfg, '--phase', 'dev', '--seed', '101', '--stage', 'ablations'])
    rd = sorted(ws.runs.glob(f'dev_{cfg}_s101_*'), key=lambda p: p.stat().st_mtime)[-1]
    run(['predict.py', '--run', rd.name, '--partitions', 'tune', '--views', 'full'])
# common shorter-schedule pilots (configs 7 and 8 of the eight allowed): complete 3-epoch cosine schedule for P and the pretrained comparator
for cfg in ['P', 'B3']:
    run(['train.py', '--config', cfg, '--phase', 'dev', '--seed', '101', '--epochs', '3', '--stage', 'ablations', '--tag', 'ep3'])
    rd = sorted(ws.runs.glob(f'dev_{cfg}_s101_*'), key=lambda p: p.stat().st_mtime)[-1]
    run(['predict.py', '--run', rd.name, '--partitions', 'tune', '--views', 'full'])

run dev_A1_s101_4455edaa_2406ce59_cb621ea3 already PASS; reuse (hashes match)


dev_A1_s101_4455edaa_2406ce59_cb621ea3 tune full: 10667 rows in 1.7s -> tune_full_best.parquet; finite=True


run dev_A2_s101_4455edaa_2406ce59_0a1ef11a already PASS; reuse (hashes match)


dev_A2_s101_4455edaa_2406ce59_0a1ef11a tune full: 10667 rows in 1.7s -> tune_full_best.parquet; finite=True


run dev_P_s101_4455edaa_2406ce59_c6dac984 already PASS; reuse (hashes match)


...

dev_P_MSF_s101_4455edaa_2406ce59_4611f609 tune full: 10667 rows in 1.9s -> tune_full_best.parquet; finite=True


run dev_P_s101_4455edaa_2406ce59_385fb705 already PASS; reuse (hashes match)


dev_P_s101_4455edaa_2406ce59_385fb705 tune full: 10667 rows in 1.7s -> tune_full_best.parquet; finite=True


run dev_B3_s101_4455edaa_2406ce59_1468c89c already PASS; reuse (hashes match)


dev_B3_s101_4455edaa_2406ce59_1468c89c tune full: 10667 rows in 1.2s -> tune_full_best.parquet; finite=True


## Development table, comparator selection, feasibility scenarios
All compact variants and the pretrained comparator reach their best tune loss within the first three epochs of the 12-epoch schedule, so the allowed *common shorter epoch count* pilot (a complete 3-epoch cosine schedule) is run for P and B3. Selection is restricted to runs trained with that complete schedule, because the final refit must run a complete schedule without tune-based checkpoint selection.

In [3]:
run(['evaluate_dev.py', '--schedule-epochs', '3'])
cand = read_json(ws.evaluation / 'development' / 'protocol_candidate.json'); print(json.dumps({k: cand.get(k) for k in ['comparator','candidate_run','final_epochs','development_paired_sd','mde_test_at_dev_sd','mde_scenarios','referral_score','referral_score_aurc_tune','n_development_configs']}, indent=1))

config  seed status  epochs  lr_mult  best_epoch  tune_patient_kl  tune_row_kl  tune_macro_auroc  wall_min  params
    A1   101   PASS    12.0      1.0         2.0         1.001073     0.955482          0.832881       4.6  147524
    A2   101   PASS    12.0      1.0         1.0         0.990307     0.954771          0.844408       4.7  155877
    B0     0   PASS     NaN      NaN         NaN         1.410066     1.397289               NaN       NaN       0
    B1     0   PASS     NaN      NaN         NaN         1.240545     1.220292               NaN       NaN     450
    B2   101   PASS    12.0      1.0         2.0         0.986238     0.958555          0.841038       4.4  147524
    B3   101   PASS     3.0      1.0         3.0         0.926608     0.875098          0.885384       1.1 1524150
    B3   101   PASS    12.0      1.0         3.0         0.895890     0.899257          0.874065       4.8 1524150
     P   101   PASS     3.0      1.0         3.0         0.926970     0.901546  

## Input-only feature audit and parameter counts

In [4]:
from cape_eeg.model import build_model, count_parameters, CONFIGS
for cid in list(CONFIGS) + ['B3']:
    print(cid, count_parameters(build_model(cid)) if cid != 'B3' else 'see notebook 02')
print('gate inputs: [uL, uC, valid_L, valid_C, JS(pL,pC)] - no patient id, vote count or label enters the network at inference')

B2 {'total': 147524, 'trainable': 147524}
A1 {'total': 147524, 'trainable': 147524}
A2 {'total': 155877, 'trainable': 155877}
P {'total': 164134, 'trainable': 164134}
P_MSF {'total': 208070, 'trainable': 208070}
B3 see notebook 02
gate inputs: [uL, uC, valid_L, valid_C, JS(pL,pC)] - no patient id, vote count or label enters the network at inference


## Leakage audit rerun and protocol lock
The lock is written once; it binds the comparator, epochs, referral score, strata, corruption list and figure list before any test access.

In [5]:
run(['make_splits.py'])  # deterministic: must reproduce the identical split hash
assert read_json(ws.manifests / 'leakage_audit.json')['status'] == 'PASS'
if not (ws.manifests / 'protocol_lock.json').exists():
    run(['evaluate_dev.py', '--schedule-epochs', '3', '--lock'])
lock = read_json(ws.manifests / 'protocol_lock.json'); print('protocol_hash', lock['protocol_hash'], '| comparator', lock['comparator'], '| final epochs', lock['final_epochs'], '| referral score', lock['referral_score'])

train          rows  64292 (0.602) patients 1170 comps 1170 class share [0.125, 0.14, 0.164, 0.143, 0.162, 0.267]
tune           rows  10667 (0.100) patients  196 comps  196 class share [0.109, 0.178, 0.196, 0.12, 0.129, 0.267]
calibration_t  rows   5289 (0.050) patients   95 comps   95 class share [0.11, 0.183, 0.17, 0.137, 0.122, 0.279]
calibration_p  rows   5247 (0.049) patients   98 comps   98 class share [0.108, 0.171, 0.206, 0.11, 0.118, 0.288]
test           rows  21305 (0.200) patients  391 comps  391 class share [0.121, 0.184, 0.186, 0.107, 0.124, 0.278]
split_hash 4455edaac7cb7e5d leakage PASS forbidden 0
protocol_hash 380269a4a6db7334 | comparator B3 | final epochs 3 | referral score entropy
